### Titanic Survival Classification: End-to-End Baseline Pipeline

An end-to-end ML pipeline built on the OpenML Titanic dataset (`v1`), demonstrating data cleaning, preprocessing with Scikit-Learn `Pipeline` + `ColumnTransformer`, and baseline model metric evaluation.

#### Workflow Overview

* **Data Cleaning & Leakage Prevention:** Dropped post-disaster leakage features (`boat`, `body`, `home.dest`) and high-cardinality text fields (`cabin`, `name`, `ticket`), standardised categorical strings (`sex`, `embarked`), and switched passenger class (`pclass`) to a categorical variable.
* **Train/Test Split:** Used `StratifiedShuffleSplit` (80/20) prior to processing to preserve class balance and prevent data leakage.
* **Preprocessing Pipeline:**
* **Numeric (`age`, `fare`, `sibsp`, `parch`):** Median imputation followed by `StandardScaler`.
* **Categorical (`sex`, `embarked`, `pclass`):** Most-frequent imputation followed by `OneHotEncoder(handle_unknown='ignore')`.


* **Evaluation:** Evaluated both models using 5-fold cross-validation (`cross_validate`) and a held-out test set across Accuracy, Precision, Recall, and F1-Score.

#### Results

| Model | CV Accuracy | CV Precision | CV Recall | CV F1 | Test Accuracy | Test Precision | Test Recall | Test F1 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **Logistic Regression** | 0.775 | 0.715 | 0.683 | 0.697 | **0.805** | **0.763** | 0.710 | **0.736** |
| **Random Forest** | 0.773 | 0.714 | 0.678 | 0.693 | 0.790 | 0.723 | **0.730** | 0.726 |

#### Key Takeaways

* **Clean Generalization:** Both models achieved slightly higher accuracy and F1 scores on the unseen test set (~79%–80.5%) than on 5-fold cross-validation (~77.3%–77.5%) --> Preprocessing pipeline avoided overfitting and data leakage.
* **Linear vs. Tree Baseline:** Logistic Regression slightly outperformed the untuned Random Forest across overall accuracy (`80.5%` vs. `79.0%`) and F1 (`0.736` vs. `0.726`).
* **Precision vs. Recall Tradeoff:** Logistic Regression produced fewer false-positive survival predictions (`0.763` test precision), whereas Random Forest captured slightly more actual survivors (`0.730` test recall).

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

#Splitting + cross-validation
from sklearn.model_selection import cross_validate, StratifiedShuffleSplit

#Processing
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline

#Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

#Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [15]:
data = fetch_openml("titanic", version =1, as_frame=True).frame
data['survived'] = pd.to_numeric(data['survived'])
print(data['survived'].value_counts())
print()
print()
print(data.isna().sum())
data = data.drop(columns=['boat', 'body', 'home.dest', 'cabin', 'name', 'ticket'])

survived
0    809
1    500
Name: count, dtype: int64


pclass          0
survived        0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64


In [16]:
X = data.drop('survived', axis = 1)
X['sex'] = X['sex'].str.strip().str.lower()
X['embarked'] = X['embarked'].str.strip().str.lower()
y = data['survived']

split = StratifiedShuffleSplit(n_splits = 1, test_size = 0.2, random_state= 42)

for train_idx, test_idx in split.split(X,y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


In [20]:
num_columns = X_train.select_dtypes(include='number').columns.tolist()
cat_columns = X_train.select_dtypes(exclude='number').columns.tolist()



# Passenger ticket class switched to category columns
num_columns.remove('pclass')
cat_columns.append('pclass')


# processing, scaling and encoding. To pass onto each model.
preprocessor = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy= 'median'), StandardScaler()), num_columns),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), cat_columns),
    ],
)


log_reg_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression())
])

rf_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(random_state=42))
])

models = {
    'Logistic Regression': log_reg_pipe,
    'Random Forest' : rf_pipe
}

metrics = ['accuracy', 'precision', 'recall', 'f1']
results = {}
for name, pipe in models.items():
    scores = cross_validate(pipe, X_train, y_train, cv=5, scoring=metrics)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results[name] = {
        **{m: scores[f'test_{m}'].mean() for m in metrics},
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_precision': precision_score(y_test, y_pred),
        'test_recall': recall_score(y_test, y_pred),
        'test_f1': f1_score(y_test, y_pred),
    }

    

results_df = pd.DataFrame(results).T
print(results_df.to_string())

                     accuracy  precision  recall        f1  test_accuracy  test_precision  test_recall   test_f1
Logistic Regression  0.774577   0.714716  0.6825  0.696933       0.805344        0.763441         0.71  0.735751
Random Forest        0.772627   0.714160  0.6775  0.693351       0.790076        0.722772         0.73  0.726368
